# NP3-561-CD — Seven-Day Load Forecast by Weather Zone — Endpoint Check

**Report:** NP3-561-CD (Report Type ID 12312), API operation `getData_7d_load_fcast_by_wzn`.

**Goal of this check (before writing the full extractor):**
1. Confirm the live endpoint does **not** cover Aug 2023 (needs the archive).
2. Confirm the archive list + batch-download + nested-unzip flow works.
3. Run a **timed one-day sample extraction** (delivery date 2023-08-15, keeping every posting) with the 5 selected columns.
4. Extrapolate the one-day wall time to a full-month runtime estimate.

**Endpoints**
- Live: `GET  https://api.ercot.com/api/public-reports/np3-561-cd/7d_load_fcast_by_wzn`
- Archive list: `GET  https://api.ercot.com/api/public-reports/archive/np3-561-cd?postDatetimeFrom=..&postDatetimeTo=..`
- Archive download: `POST https://api.ercot.com/api/public-reports/archive/np3-561-cd/download` body `{"docIds":[..]}`

**Selected output columns:** `deliveryDate, hourEnding, systemTotal, DSTFlag, postedDatetime` (postedDatetime injected from archive doc metadata — it is not in the CSV).

In [2]:
import io, sys, time, zipfile
from pathlib import Path
from datetime import datetime, timedelta
import pandas as pd, requests

# Reuse the repo's auth/client (ROPC id_token + subscription key).
sys.path.insert(0, str(Path.cwd().parents[1] / '02_scripts' / '1_scrapers'))
from ercot_common import ERCOTAuth, ERCOTAPIClient, require_env
import logging; logging.basicConfig(level=logging.WARNING); log = logging.getLogger('check')

# NOTE: the archive POST /download route is CASE-SENSITIVE and needs the UPPERCASE
# EMIL id (NP3-561-CD). Lowercase POST -> HTTP 405. The list GET accepts either case.
LIVE     = 'https://api.ercot.com/api/public-reports/np3-561-cd/7d_load_fcast_by_wzn'
ARCHIVE  = 'https://api.ercot.com/api/public-reports/archive/NP3-561-CD'
DOWNLOAD = f'{ARCHIVE}/download'

auth   = ERCOTAuth(require_env('ERCOT_USERNAME', log), require_env('ERCOT_PASSWORD', log), log)
client = ERCOTAPIClient(auth, require_env('ERCOT_SUBSCRIPTION_KEY', log), log)
print('Auth + client ready.')

Auth + client ready.


## 1. Live endpoint returns 0 records for Aug 2023 — archive required

In [3]:
rows, meta = client.fetch_page(LIVE, {'deliveryDateFrom': '2023-08-01', 'deliveryDateTo': '2023-08-31'}, 1, 1)
print('Live endpoint Aug 2023  -> totalRecords =', meta.get('totalRecords'))

# Field metadata (self-describing GET with no params)
fields = requests.get(LIVE, headers=client._headers(), timeout=(10, 60)).json().get('fields', [])
print('Schema fields:', [f['name'] for f in fields])

Live endpoint Aug 2023  -> totalRecords = 0
Schema fields: ['postedDatetime', 'deliveryDate', 'hourEnding', 'coast', 'east', 'farWest', 'north', 'northCentral', 'southCentral', 'southern', 'west', 'systemTotal', 'DSTFlag']


## 2. Archive list for one delivery day (2023-08-15)

To capture *every* posting that forecasts delivery date 2023-08-15, we list postings from 8 days earlier (an 8-day forecast posted on 08-07 already covers 08-15) through 08-15.

In [4]:
def list_docs(ts_from, ts_to, page_size=1000):
    """Return {docId: postDatetime} for a postDatetime window."""
    docs, page = {}, 1
    while True:
        p = {'postDatetimeFrom': ts_from, 'postDatetimeTo': ts_to, 'page': page, 'size': page_size}
        r = client._session.get(ARCHIVE, headers=client._headers(), params=p, timeout=(10, 60))
        r.raise_for_status(); b = r.json()
        for a in b.get('archives', []):
            docs[int(a['docId'])] = pd.to_datetime(a['postDatetime'])
        if page >= int(b.get('_meta', {}).get('totalPages', 1)):
            break
        page += 1; time.sleep(1)
    return docs

DELIVERY_DAY = '2023-08-15'
ts_from = (datetime.strptime(DELIVERY_DAY, '%Y-%m-%d') - timedelta(days=8)).strftime('%Y-%m-%dT00:00:00')
ts_to   = f'{DELIVERY_DAY}T23:59:59'
doc_map = list_docs(ts_from, ts_to)
print(f'Postings listed for {ts_from} -> {ts_to}: {len(doc_map)} docs')
print('postDatetime range:', min(doc_map.values()), '->', max(doc_map.values()))

Postings listed for 2023-08-07T00:00:00 -> 2023-08-15T23:59:59: 216 docs
postDatetime range: 2023-08-07 00:30:00 -> 2023-08-15 23:30:04


## 3. Timed one-day sample extraction (all postings, 5 columns)

In [9]:
COLUMN_RENAMES = {'DeliveryDate': 'deliveryDate', 'HourEnding': 'hourEnding',
                  'SystemTotal': 'systemTotal', 'DSTFlag': 'DSTFlag'}
OUTPUT_COLS = ['deliveryDate', 'hourEnding', 'systemTotal', 'DSTFlag', 'postedDatetime']
BATCH = 50  # archive POST /download caps at ~50 docIds; n>=100 -> HTTP 405.

def parse_csv(raw, posted):
    df = pd.read_csv(io.BytesIO(raw), dtype=str)
    df = df.rename(columns=COLUMN_RENAMES)[['deliveryDate', 'hourEnding', 'systemTotal', 'DSTFlag']].copy()
    df['deliveryDate'] = pd.to_datetime(df['deliveryDate'], format='%m/%d/%Y', errors='coerce').dt.strftime('%Y-%m-%d')
    df['systemTotal']  = pd.to_numeric(df['systemTotal'], errors='coerce')
    df['DSTFlag']      = df['DSTFlag'].str.strip().str.upper().map({'Y': True, 'N': False}).astype('boolean')
    df['postedDatetime'] = posted
    return df

def download_batch(doc_ids):
    h = {**client._headers(), 'Content-Type': 'application/json', 'Accept': 'application/zip'}
    r = client._session.post(DOWNLOAD, headers=h, json={'docIds': doc_ids}, timeout=(10, 240))
    r.raise_for_status(); return r.content

def process_zip(zip_bytes, doc_map):
    frames = []
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as outer:
        for name in outer.namelist():
            docid  = int(name.split('.', 1)[0])          # outer entry begins with docId
            posted = doc_map.get(docid)
            with zipfile.ZipFile(io.BytesIO(outer.read(name))) as inner:
                for cn in [n for n in inner.namelist() if n.lower().endswith('.csv')]:
                    frames.append(parse_csv(inner.read(cn), posted))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=OUTPUT_COLS)

def fetch_all(doc_map):
    """Download every docId in chunks of BATCH, tag postedDatetime, concat."""
    ids, frames = sorted(doc_map), []
    for i in range(0, len(ids), BATCH):
        frames.append(process_zip(download_batch(ids[i:i + BATCH]), doc_map))
        time.sleep(0.25)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=OUTPUT_COLS)

# --- timed sample extraction for one delivery day ---
t0 = time.perf_counter()
ids = sorted(doc_map)
raw = fetch_all(doc_map)
sample = raw[raw['deliveryDate'] == DELIVERY_DAY][OUTPUT_COLS] \
            .drop_duplicates(subset=['deliveryDate', 'hourEnding', 'postedDatetime']).reset_index(drop=True)
elapsed = time.perf_counter() - t0

print(f'Elapsed for {len(ids)} postings -> deliveryDate {DELIVERY_DAY}: {elapsed:.1f}s')
print(f'Rows kept (all postings): {len(sample):,}')
print(f'Distinct postings covering the day: {sample["postedDatetime"].nunique()}')
print(f'Delivery-hours: {sample["hourEnding"].nunique()} (expect 24)')
print(f'systemTotal nulls: {int(sample["systemTotal"].isna().sum())}')
print('\nDtypes:'); print(sample.dtypes)

Elapsed for 216 postings -> deliveryDate 2023-08-15: 5.6s
Rows kept (all postings): 4,608
Distinct postings covering the day: 192
Delivery-hours: 24 (expect 24)
systemTotal nulls: 0

Dtypes:
deliveryDate                 str
hourEnding                   str
systemTotal              float64
DSTFlag                  boolean
postedDatetime    datetime64[us]
dtype: object


In [10]:
sample

,deliveryDate,hourEnding,systemTotal,DSTFlag,postedDatetime
0,2023-08-15,1:00,63820.5489,False,2023-08-08 00:30:04
1,2023-08-15,2:00,60521.5586,False,2023-08-08 00:30:04
2,2023-08-15,3:00,57935.6094,False,2023-08-08 00:30:04
3,2023-08-15,4:00,56504.0810,False,2023-08-08 00:30:04
4,2023-08-15,5:00,55883.3390,False,2023-08-08 00:30:04
...,...,...,...,...,...
4603,2023-08-15,20:00,76240.0894,False,2023-08-15 23:30:04
4604,2023-08-15,21:00,72808.5909,False,2023-08-15 23:30:04
4605,2023-08-15,22:00,69694.7006,False,2023-08-15 23:30:04
4606,2023-08-15,23:00,65600.8310,False,2023-08-15 23:30:04


## 4. Runtime extrapolation to the full August pull

In [ ]:
# Full Aug pull lists postDatetime 2023-07-24 -> 2023-08-31 (~39 days x ~24 postings/day).
days_full   = 39
postings_day = len(doc_map) / 9.0          # one-day sample covered a 9-day posting window
est_postings = postings_day * days_full
# Wall time scales ~linearly with #postings (download bytes + parse dominate).
est_seconds = elapsed * est_postings / len(ids)
print(f'~postings/day        : {postings_day:.0f}')
print(f'est. postings (Aug)  : {est_postings:,.0f}')
print(f'est. full-month time : {est_seconds/60:.1f} min  -> {"Method B (local)" if est_seconds < 600 else "Method C (HPS)"}')

~postings/day        : 24
est. postings (Aug)  : 936
est. full-month time : 0.4 min  -> Method B (local)


## Findings

- Live endpoint: **0 records** for Aug 2023 — archive is required.
- Archive flow verified: list (`docId` → `postDatetime`) → `POST /download` → outer ZIP → inner ZIP → CSV.
- **Two archive quirks discovered (both matter for the extractor):**
  - The `POST /download` route is **case-sensitive** — needs the **UPPERCASE** EMIL (`NP3-561-CD`); lowercase → HTTP 405.
  - `POST /download` **caps at ~50 docIds per request** — `n=50` → 200, `n≥100` → HTTP 405. Chunk into batches of 50.
- Archive CSV is Title_Case, 192 rows/posting (8-day forecast); `postedDatetime` is injected from the doc metadata via the outer-ZIP entry's leading `docId`.
- One-day sample extraction succeeds with the 5 selected columns; see the timing cell for wall time and the full-month extrapolation -> use Method B."